# [SOLUTION] UdaPlay Project

## Part 01 - Offline RAG

UdaPlay is an AI research agent for the video game industry. This first notebook builds
the knowledge base the agent will rely on:

1. Load and validate the game records in `games/*.json`
2. Turn each record into a document + metadata pair
3. Embed everything into a **persistent** ChromaDB collection
4. Show that the collection answers semantic queries, with and without metadata filters
5. Wrap it all in a reusable vector-store manager that Part 02 imports

The dataset ships with 15 games and has been extended to 25 (see `games/016.json`
onwards) so the agent has a little more to work with.

### Setup

In [1]:
# Only needed for Udacity workspace
# ChromaDB needs a newer sqlite3 than the workspace image provides.

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
import os
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

# Shared helpers so Part 01 and Part 02 always use the same database,
# the same collection names and the same embedding model.
from lib.udaplay import (
    CHROMA_PATH,
    GAMES_COLLECTION,
    GAMES_DIR,
    GameVectorStore,
    format_game_document,
    game_metadata,
    get_embedding_function,
    load_config,
    load_games,
)

print("chromadb", chromadb.__version__)

chromadb 1.5.9


### Environment

Create a file named `config.env` next to this notebook (a `.env` file works too):

```
OPENAI_API_KEY="voc-**********"
TAVILY_API_KEY="tvly-*********"
OPENAI_BASE_URL="https://openai.vocareum.com/v1"
```

`load_config()` reads it, checks both keys are present, and makes sure the base URL is
exported. The OpenAI SDK reads `OPENAI_BASE_URL` from the environment by itself, which
is what lets `lib/llm.py` talk to the Vocareum proxy without any changes. ChromaDB does
**not** read that variable, so the base URL is passed to the embedding function
explicitly in the next section.

In [3]:
config = load_config()

Credentials loaded from config.env
  OPENAI_API_KEY  : voc-21...9965 (49 chars)
  TAVILY_API_KEY  : tvly-d...siQE (58 chars)
  OPENAI_BASE_URL : https://openai.vocareum.com/v1


### VectorDB Instance

A `PersistentClient` writes to disk, so the embeddings survive a kernel restart and
Part 02 can open the same collection instead of re-embedding everything.

In [4]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

print(f"Persistent ChromaDB at: {Path(CHROMA_PATH).resolve()}")
print("Existing collections:", [col.name for col in chroma_client.list_collections()])

Persistent ChromaDB at: /Users/akaile/Development/Agentic AI Course/Udaplay/solution/chromadb
Existing collections: []


### Embedding function

The embedding model is the one decision that has to be identical everywhere. Open a
collection with a different model later and retrieval either silently degrades or fails
outright on a vector-dimension mismatch, so the choice lives in one place
(`lib/udaplay.py`) and both notebooks call it.

`get_embedding_function()` tries `text-embedding-3-small` first, falls back to
`text-embedding-ada-002` if the endpoint does not serve it, and only as a last resort
uses ChromaDB's bundled local model. Each candidate is smoke-tested with a real
embedding call, so a bad key surfaces here rather than halfway through ingestion.

In [5]:
embedding_fn = get_embedding_function()

Embedding function: OpenAI `text-embedding-3-small` (1536 dimensions)


### Collection

The collection is recreated from scratch (`reset = True`) so re-running this notebook
never leaves stale documents behind. `hnsw:space=cosine` matters: with cosine distance,
`similarity = 1 - distance` lands in a predictable 0-1 range, which the agent's
evaluation tool in Part 02 uses as a first-pass quality signal.

In [6]:
reset = True

if reset:
    try:
        chroma_client.delete_collection(GAMES_COLLECTION)
        print(f"Dropped existing `{GAMES_COLLECTION}` collection")
    except Exception:
        print(f"No existing `{GAMES_COLLECTION}` collection to drop")

collection = chroma_client.get_or_create_collection(
    name=GAMES_COLLECTION,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

print(f"Collection `{collection.name}` ready - {collection.count()} documents")

No existing `udaplay` collection to drop
Collection `udaplay` ready - 0 documents


### Load and process the game data

Every file in `games/` becomes one document. The file stem (`001`, `002`, ...) is used
as the document id, which makes ingestion idempotent: re-running upserts over the same
ids instead of creating duplicates, and it gives the agent a short, stable handle to
cite in its answers.

`load_games()` also validates that each record has all six expected fields, so a
malformed file fails loudly at load time instead of producing a document with a
`None` in the middle of it.

In [7]:
games = load_games(GAMES_DIR)

print(f"Loaded {len(games)} game records from `{GAMES_DIR}/`\n")
print(f"{'id':<5} {'Name':<38} {'Platform':<22} {'Year':<6} Publisher")
print("-" * 110)
for game in games:
    print(f"{game['id']:<5} {game['Name'][:37]:<38} {game['Platform'][:21]:<22} "
          f"{game['YearOfRelease']:<6} {game['Publisher']}")

Loaded 25 game records from `games/`

id    Name                                   Platform               Year   Publisher
--------------------------------------------------------------------------------------------------------------
001   Gran Turismo                           PlayStation 1          1997   Sony Computer Entertainment
002   Grand Theft Auto: San Andreas          PlayStation 2          2004   Rockstar Games
003   Gran Turismo 5                         PlayStation 3          2010   Sony Computer Entertainment
004   Marvel's Spider-Man                    PlayStation 4          2018   Sony Interactive Entertainment
005   Marvel's Spider-Man 2                  PlayStation 5          2023   Sony Interactive Entertainment
006   Pokémon Gold and Silver                Game Boy Color         1999   Nintendo
007   Pokémon Ruby and Sapphire              Game Boy Advance       2002   Nintendo
008   Super Mario World                      Super Nintendo Entert  1990   Nintendo
009   

#### Document text vs. metadata

Two different things are stored per game, and the split matters:

* **Document** - the text that actually gets embedded. Every field goes in, not just the
  description. A question like *"which games did Nintendo publish on the Game Boy
  Color?"* only matches semantically if the publisher and platform are part of the
  vector.
* **Metadata** - the same fields kept as structured scalars, so they can be used for
  exact `where` filters and returned to the agent as clean JSON rather than parsed back
  out of prose.

In [8]:
example = games[5]

print("RAW RECORD")
print(json.dumps({k: v for k, v in example.items() if k != "id"}, indent=2, ensure_ascii=False))
print("\nDOCUMENT TO EMBED")
print(format_game_document(example))
print("\nMETADATA")
print(json.dumps(game_metadata(example), indent=2, ensure_ascii=False))

RAW RECORD
{
  "Name": "Pokémon Gold and Silver",
  "Platform": "Game Boy Color",
  "Genre": "Role-playing",
  "Publisher": "Nintendo",
  "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.",
  "YearOfRelease": 1999
}

DOCUMENT TO EMBED
Pokémon Gold and Silver (1999) - Platform: Game Boy Color. Genre: Role-playing. Publisher: Nintendo. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.

METADATA
{
  "Name": "Pokémon Gold and Silver",
  "Platform": "Game Boy Color",
  "Genre": "Role-playing",
  "Publisher": "Nintendo",
  "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.",
  "YearOfRelease": 1999
}


### Add documents

In [9]:
ids = [game["id"] for game in games]
documents = [format_game_document(game) for game in games]
metadatas = [game_metadata(game) for game in games]

# upsert (not add) so this cell is safe to re-run
collection.upsert(ids=ids, documents=documents, metadatas=metadatas)

print(f"Indexed {len(ids)} documents - collection now holds {collection.count()}")

Indexed 25 documents - collection now holds 25


### Semantic search

The point of the vector store is that queries do not have to share vocabulary with the
documents. None of the questions below use the words that appear in the records, and the
similarity score shows how confident the match is.

In [10]:
def search(query: str, n_results: int = 3, where: dict = None):
    """Query the collection and print the hits with their similarity scores."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    print(f"Q: {query}")
    if where:
        print(f"   filter: {where}")
    if not results["ids"][0]:
        print("   (no matches)\n")
        return results

    for doc_id, metadata, distance in zip(
        results["ids"][0], results["metadatas"][0], results["distances"][0]
    ):
        similarity = max(0.0, 1.0 - distance)
        print(f"   [{doc_id}] sim={similarity:.3f}  {metadata['Name']} "
              f"({metadata['YearOfRelease']}, {metadata['Platform']}) - {metadata['Publisher']}")
    print()
    return results


_ = search("When was Pokémon Gold and Silver released?")
_ = search("Which one was the first 3D platformer Mario game?")
_ = search("a racing simulator for the original PlayStation")
_ = search("open world game where you play an outlaw in the old west")

Q: When was Pokémon Gold and Silver released?
   [006] sim=0.692  Pokémon Gold and Silver (1999, Game Boy Color) - Nintendo
   [017] sim=0.556  Pokémon Red and Blue (1996, Game Boy) - Nintendo
   [007] sim=0.543  Pokémon Ruby and Sapphire (2002, Game Boy Advance) - Nintendo

Q: Which one was the first 3D platformer Mario game?
   [009] sim=0.632  Super Mario 64 (1996, Nintendo 64) - Nintendo
   [008] sim=0.572  Super Mario World (1990, Super Nintendo Entertainment System (SNES)) - Nintendo
   [010] sim=0.390  Super Smash Bros. Melee (2001, GameCube) - Nintendo

Q: a racing simulator for the original PlayStation
   [001] sim=0.629  Gran Turismo (1997, PlayStation 1) - Sony Computer Entertainment
   [003] sim=0.562  Gran Turismo 5 (2010, PlayStation 3) - Sony Computer Entertainment
   [025] sim=0.398  Forza Horizon 5 (2021, Xbox Series X|S) - Xbox Game Studios

Q: open world game where you play an outlaw in the old west
   [021] sim=0.482  Red Dead Redemption 2 (2018, PlayStation 4) - Ro

#### Where semantic search runs out

Mortal Kombat X is not in the dataset. The query still returns the three *closest*
documents, because vector search always returns its nearest neighbours - it has no
concept of "I do not know this".

That is exactly why Part 02 needs an explicit evaluation step: something has to look at
these results and decide they do not answer the question before falling back to the web.

In [11]:
_ = search("Was Mortal Kombat X released for PlayStation 5?")

Q: Was Mortal Kombat X released for PlayStation 5?
   [018] sim=0.472  God of War Ragnarök (2022, PlayStation 5) - Sony Interactive Entertainment
   [005] sim=0.463  Marvel's Spider-Man 2 (2023, PlayStation 5) - Sony Interactive Entertainment
   [025] sim=0.449  Forza Horizon 5 (2021, Xbox Series X|S) - Xbox Game Studios



### Metadata filtering

Semantic search answers fuzzy questions; metadata filters answer precise ones. Combining
them narrows the search to a subset before ranking by similarity.

In [12]:
_ = search("role playing games", n_results=3, where={"Publisher": {"$eq": "Nintendo"}})
_ = search("racing", n_results=3, where={"YearOfRelease": {"$gte": 2015}})
_ = search("shooter", n_results=2, where={"Platform": {"$eq": "Xbox Series X|S"}})

Q: role playing games
   filter: {'Publisher': {'$eq': 'Nintendo'}}
   [017] sim=0.338  Pokémon Red and Blue (1996, Game Boy) - Nintendo
   [007] sim=0.334  Pokémon Ruby and Sapphire (2002, Game Boy Advance) - Nintendo
   [006] sim=0.312  Pokémon Gold and Silver (1999, Game Boy Color) - Nintendo

Q: racing
   filter: {'YearOfRelease': {'$gte': 2015}}
   [025] sim=0.354  Forza Horizon 5 (2021, Xbox Series X|S) - Xbox Game Studios
   [012] sim=0.269  Mario Kart 8 Deluxe (2017, Nintendo Switch) - Nintendo
   [021] sim=0.187  Red Dead Redemption 2 (2018, PlayStation 4) - Rockstar Games

Q: shooter
   filter: {'Platform': {'$eq': 'Xbox Series X|S'}}
   [015] sim=0.288  Halo Infinite (2021, Xbox Series X|S) - Xbox Game Studios
   [025] sim=0.159  Forza Horizon 5 (2021, Xbox Series X|S) - Xbox Game Studios



### A reusable vector store manager

Raw ChromaDB results come back as `{'ids': [[...]], 'distances': [[...]]}` - nested
lists that every caller has to unpack the same way. `GameVectorStore` (in
`lib/udaplay.py`) wraps the client and returns typed `RetrievedGame` objects with the
distance already converted to a similarity score.

Part 02 builds the `retrieve_game` tool directly on top of this class, so the agent and
this notebook cannot drift apart.

In [13]:
store = GameVectorStore(reset=False)   # opens the collection created above

hits = store.search("Who published Halo Infinite and when did it come out?", n_results=3)

for hit in hits:
    print(f"[{hit.source_id}] sim={hit.similarity:.3f}  {hit.Name} ({hit.YearOfRelease})")
    print(f"        platform : {hit.Platform}")
    print(f"        publisher: {hit.Publisher}")
    print(f"        genre    : {hit.Genre}")
print("\nAs JSON (this is the shape the agent's tool returns):")
print(hits[0].model_dump_json(indent=2))

Embedding function: OpenAI `text-embedding-3-small` (1536 dimensions)
Collection `udaplay` ready with 25 documents
[015] sim=0.663  Halo Infinite (2021)
        platform : Xbox Series X|S
        publisher: Xbox Game Studios
        genre    : First-person shooter
[014] sim=0.376  Minecraft (2014)
        platform : Xbox One
        publisher: Mojang Studios
        genre    : Sandbox, Survival
[016] sim=0.325  The Legend of Zelda: Breath of the Wild (2017)
        platform : Nintendo Switch
        publisher: Nintendo
        genre    : Action-adventure

As JSON (this is the shape the agent's tool returns):
{
  "source_id": "015",
  "Name": "Halo Infinite",
  "Platform": "Xbox Series X|S",
  "YearOfRelease": 2021,
  "Genre": "First-person shooter",
  "Publisher": "Xbox Game Studios",
  "Description": "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.",
  "similarity": 0.6633
}


### Persistence check

Opening a brand-new client against the same path proves the embeddings are on disk and
not just in this kernel's memory. The embedding function is passed again on `get` -
ChromaDB needs it to embed future *queries*, not just the stored documents.

In [14]:
fresh_client = chromadb.PersistentClient(path=CHROMA_PATH)
fresh_collection = fresh_client.get_collection(
    name=GAMES_COLLECTION,
    embedding_function=embedding_fn,
)

print(f"Reopened `{GAMES_COLLECTION}` from disk: {fresh_collection.count()} documents")

probe = fresh_collection.query(
    query_texts=["Nintendo Switch kart racing game"],
    n_results=2,
    include=["metadatas", "distances"],
)
for doc_id, metadata, distance in zip(probe["ids"][0], probe["metadatas"][0], probe["distances"][0]):
    print(f"  [{doc_id}] sim={1 - distance:.3f}  {metadata['Name']} ({metadata['YearOfRelease']})")

Reopened `udaplay` from disk: 25 documents
  [012] sim=0.649  Mario Kart 8 Deluxe (2017)
  [025] sim=0.437  Forza Horizon 5 (2021)


### Summary

| Step | Result |
|---|---|
| Dataset | 25 game records loaded and validated from `games/*.json` |
| Documents | All six fields embedded per game; same fields kept as filterable metadata |
| Store | Persistent ChromaDB collection `udaplay` with cosine distance |
| Retrieval | Semantic search, metadata filters, and a typed `GameVectorStore` wrapper |

The gap this notebook makes visible - vector search returning confident-looking
neighbours for a game that is not in the dataset - is what Part 02 handles with an
evaluation tool and a web-search fallback.